In [1]:
# Libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
# ML Models
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR

In [3]:
# Import Dataset
df = pd.read_csv('./dataset.csv')

In [4]:
df = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

# Define categorical mappings
ordinal_mappings = {
    'Income': ['Low (Below 15,000)', 'Lower middle (15,000-30,000)', 'Upper middle (30,000-50,000)', 'High (Above 50,000)'],
    'Hometown': ['Village', 'City'],
    'Preparation': ['0-1 Hour', '2-3 Hours', 'More than 3 Hours'],
    'Gaming': ['0-1 Hour', '2-3 Hours', 'More than 3 Hours'],
    'Attendance': ['Below 40%', '40%-59%', '60%-79%', '80%-100%'],
    'Semester': ['2nd', '3rd', '4th', '5th', '6th', '7th', '8th', '9th', '10th', '11th', '12th'],
    'Job': ['No', 'Yes'],
    'Extra': ['No', 'Yes']
}

# Apply ordinal encoding
for column, categories in ordinal_mappings.items():
    df[column] = OrdinalEncoder(categories=[categories]).fit_transform(df[[column]])

In [5]:
# Function for OneHotEncoding
def one_hot_encode(df, columns):
    encoder = OneHotEncoder(sparse_output=False)
    encoded = encoder.fit_transform(df[columns])
    df_encoded = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(columns))
    return pd.concat([df.drop(columns, axis=1), df_encoded], axis=1)

# Apply OneHotEncoding to 'Department' and 'Gender'
df = one_hot_encode(df, ['Department', 'Gender'])

In [6]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_selection import RFE
import numpy as np

def train_best_model(df, threshold_range=(0, 1, 0.01)):
    # Define target and features
    X = df.drop(columns=['Overall'])
    y = df['Overall']
    
    # Initialize the columns to keep
    columns_to_keep_X = {'Income', 'Hometown', 'Job'}
    
    # Initialize variable to track the best threshold and performance for each model
    results = {}
    
    # List of models
    models = {
        'linear_regression': LinearRegression(),
        'decision_tree': DecisionTreeRegressor(random_state=42),
        'random_forest': RandomForestRegressor(random_state=42),
        'svr': SVR()
    }
    
    for model_name, model in models.items():
        best_threshold = None
        best_rmse = float('inf')
        best_r2 = float('-inf')
        
        for threshold in np.arange(*threshold_range):
            # Identify columns to drop for X, keeping specific features
            columns_to_drop_X = [col for col in X.columns if col not in columns_to_keep_X]
            
            # Update X by dropping the selected columns
            X_filtered = X.drop(columns=columns_to_drop_X)
            
            # Apply RFE (Recursive Feature Elimination) to select the best features
            rfe = RFE(model, n_features_to_select=3)  # Selecting top 5 features for each model (adjustable)
            X_rfe = rfe.fit_transform(X_filtered, y)
            
            # Split data into training and testing sets
            X_train, X_test, y_train, y_test = train_test_split(X_rfe, y, test_size=0.2, random_state=42)
            
            # Train the model
            model.fit(X_train, y_train)
            
            # Predict on the test set
            y_pred = model.predict(X_test)
            
            # Calculate Mean Squared Error (MSE) and R²
            mse = mean_squared_error(y_test, y_pred)
            rmse = np.sqrt(mse)  # Root Mean Squared Error
            r2 = r2_score(y_test, y_pred)
            
            # Check if this is the best RMSE so far
            if rmse < best_rmse:
                best_rmse = rmse
                best_r2 = r2
                best_threshold = threshold
        
        # Store the best result for the current model (rounding RMSE and R² to 3 decimal places)
        results[model_name] = {
            'Best Threshold': best_threshold, 
            'Best RMSE': round(best_rmse, 3),
            'Best R²': round(best_r2, 3)
        }
    
    # Print out the results for all models
    for model_name, result in results.items():
        print(f"{model_name} - Best Threshold: {result['Best Threshold']}, Best RMSE: {result['Best RMSE']}, Best R²: {result['Best R²']}")

# Example usage
train_best_model(df)


linear_regression - Best Threshold: 0.0, Best RMSE: 0.614, Best R²: -0.009
decision_tree - Best Threshold: 0.0, Best RMSE: 0.64, Best R²: -0.098
random_forest - Best Threshold: 0.0, Best RMSE: 0.643, Best R²: -0.107
svr - Best Threshold: 0.0, Best RMSE: 0.625, Best R²: -0.048
